# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

Dataset URL: [https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In Croissant, `recordSet`, `field`, and `column` entities are referenced by their unique `@id`. We'll list all record sets and display their information for further analysis.


In [ ]:
# Explore available record sets
record_sets = list(dataset.metadata.recordSet)

print("Available Record Sets and their @id:")
for record_set in record_sets:
    print(f"- @id: {record_set['@id']} | Name: {record_set.get('name', '')}")

# List fields and columns for each record set
for record_set in record_sets:
    print(f"\nRecord set @id: {record_set['@id']}")
    fields = record_set.get('field', [])
    print("Fields and Columns:")
    for field in fields:
        print(f"  - Field @id: {field['@id']} | Name: {field.get('name', '')}")
        columns = field.get('column', [])
        for column in columns:
            print(f"    - Column @id: {column['@id']} | Name: {column.get('name', '')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

For demonstration, we will extract the main survey record set (first found) and display its columns and a sample of the data.

In [ ]:
# Extract data from each record set
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Display columns for the first record set
first_record_set_id = record_set_ids[0] if len(record_set_ids) > 0 else None
if first_record_set_id:
    print(f"Data columns for record set @id {first_record_set_id}:")
    print(dataframes[first_record_set_id].columns.tolist())
    dataframes[first_record_set_id].head()
else:
    print("No record sets found in metadata.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section applies operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

### Example: Analyze and Normalize a Numeric Field

- We'll select a numeric field (e.g., GAD-7 score, typically named 'gad_7_score' or similar; use the actual column name as per previous cell output).
- Filter records where the GAD-7 score is above a threshold (e.g., 10).
- Normalize the GAD-7 score.
- Group the data by a demographic attribute (e.g., 'gender').


In [ ]:
# Set up for EDA
# Update these variables based on actual column names printed above
record_set_id = first_record_set_id

# Example plausible column ID names based on dataset description
numeric_field_id = 'gad_7_score'  # Replace with actual @id if different
group_field_id = 'gender'         # Replace with actual @id if different

# Check available columns
df = dataframes[record_set_id]
print('Available columns:', df.columns.tolist())

# EDA: filter, normalize, group
if numeric_field_id in df.columns:
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()

    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized" ]].head())

    # Grouped analysis
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id}:")
        print(grouped_df.head())
else:
    print(f"Column '{numeric_field_id}' not found in record set '{record_set_id}'.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll plot the GAD-7 score distribution and visualize mean GAD-7 scores grouped by gender.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution for the numeric field (GAD-7 score)
if numeric_field_id in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If grouped_df exists, plot group means
    if group_field_id in df.columns:
        group_means = df.groupby(group_field_id)[numeric_field_id].mean()
        plt.figure(figsize=(6, 4))
        sns.barplot(x=group_means.index, y=group_means.values)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.show()
else:
    print(f"'{numeric_field_id}' not found for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Loaded the FAIR² Mental Health Survey Dataset using `mlcroissant`.
- Explored available record sets and fields using their Croissant `@id`.
- Extracted survey data and performed basic EDA: filtered and normalized GAD-7 scores, grouped by gender.
- Visualized distributions and group means of GAD-7 scores.
- Dataset is valuable for community health analytics and AI-ready research in mental health, with unique regional focus and demographic granularity.

*For further analysis, consider exploring additional assessment scores (PHQ-9, PSQ), investigating missing data, and applying more advanced statistical or machine learning methods.*

*Notebook generated for FAIR² dataset exploration via Croissant schema and mlcroissant library.*